# Obtener audio de una fuente permitida

Este cuaderno prepara archivos de audio para pr?cticas locales. Puedes indicar enlaces permitidos o rutas a medios que tengas derecho a usar en `config.py` (archivo local ignorado por Git) o probar con la configuraci?n vac?a incluida.

```text
Fuente propia o autorizada
          ?
Descarga opcional ? extracci?n WAV mono, 16 kHz ? datos/audio/
```

No se publican archivos de voz, v?deos ni grabaciones. Comprueba las condiciones del servicio y los derechos de cada fuente antes de descargarla.


In [ ]:
import importlib
import os
import sys
from pathlib import Path

RAIZ = Path.cwd().resolve()
if not (RAIZ / "config_example.py").exists():
    RAIZ = next((p for p in RAIZ.parents if (p / "config_example.py").exists()), RAIZ)
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

if (RAIZ / "config.py").exists():
    import config as cfg
else:
    import config_example as cfg

importlib.reload(cfg)
print("Proyecto:", cfg.RAIZ)
print("Fuentes configuradas:", len(cfg.VIDEOS))


## Configuraci?n del proyecto

`config_example.py` contiene valores p?blicos seguros y una lista vac?a. Si quieres a?adir fuentes, crea `config.py` a partir de esa plantilla y ed?talo localmente; `config.py` est? excluido del repositorio para evitar publicar tus enlaces o rutas por accidente.


In [ ]:
print("Vídeos en config.py:")
for origen in cfg.VIDEOS:
    print("  ·", origen)

## Herramientas necesarias

El cuaderno usa `yt-dlp` para consultar o descargar una fuente y FFmpeg para extraer el audio. La instalaci?n de paquetes Python se realiza en la celda siguiente; FFmpeg y `ffprobe` son programas del sistema y deben estar disponibles en `PATH`.


In [ ]:
!pip install -r requirements-audio.txt


In [ ]:
import shutil, subprocess
import yt_dlp

for herramienta in ["ffmpeg", "ffprobe"]:
    ruta = shutil.which(herramienta)
    print(f"{herramienta:8}", "✅ " + ruta if ruta else "❌ no está instalado (mira la tabla de arriba)")
print(f"{'yt-dlp':8} ✅ versión {yt_dlp.version.__version__}")

## Consultar antes de descargar

Para un enlace web se consultan primero los metadatos y las variantes disponibles. Revisa duraci?n, licencia y condiciones de uso; no descargues contenido si no tienes autorizaci?n.


In [ ]:
def ficha_youtube(url):
    """Consulta los datos de una publicaci?n sin descargarla."""
    with yt_dlp.YoutubeDL({"quiet": True, "no_warnings": True, "noplaylist": True}) as ydl:
        return ydl.extract_info(url, download=False)

info = ficha_youtube(cfg.VIDEOS[0]) if cfg.VIDEOS and cfg.VIDEOS[0].startswith("http") else None
if info is None:
    print("A?ade un enlace permitido en config.py para consultar formatos web.")
else:
    for clave in ["id", "title", "channel", "duration", "upload_date", "license"]:
        print(f"{clave:12}", info.get(clave))


Una misma publicaci?n puede ofrecer varias calidades y pistas separadas. La tabla ayuda a escoger un formato compatible y a estimar el tama?o antes de iniciar la descarga.


In [ ]:
import pandas as pd

if info is not None:
    formatos = pd.DataFrame([{
        "id": f["format_id"], "ext": f["ext"], "resoluci?n": f.get("resolution"),
        "v?deo": f.get("vcodec"), "audio": f.get("acodec"),
        "MB": round((f.get("filesize") or f.get("filesize_approx") or 0) / 1e6, 1),
    } for f in info["formats"] if f.get("vcodec") != "none" or f.get("acodec") != "none"])
    print(len(formatos), "formatos disponibles")
    display(formatos[formatos["MB"] > 0])
else:
    print("No hay metadatos web que mostrar.")


## Elegir un formato

Se prioriza una resoluci?n moderada para limitar el tama?o. Si imagen y sonido vienen separados, FFmpeg los combina en un contenedor local antes de extraer la pista de audio.


In [ ]:
def descargar_video(url, altura_max=480):
    """Descarga una fuente web autorizada y devuelve su ficha."""
    opciones = {
        "format": f"bv*[height<={altura_max}]+ba/b[height<={altura_max}]/b",
        "merge_output_format": "mp4",
        "outtmpl": str(cfg.DIR_VIDEO / "%(id)s.%(ext)s"),
        "noplaylist": True, "quiet": True, "no_warnings": True, "noprogress": True,
    }
    with yt_dlp.YoutubeDL(opciones) as ydl:
        info = ydl.extract_info(url, download=True)
    archivo = next(p for p in cfg.DIR_VIDEO.glob(f"{info['id']}.*") if p.suffix not in {".part", ".ytdl"})
    return {"id": info["id"], "titulo": info["title"], "canal": info.get("channel"),
            "duracion_s": info.get("duration"), "licencia": info.get("license") or "licencia no especificada",
            "origen": url, "archivo": archivo.name}

ficha = descargar_video(cfg.VIDEOS[0]) if cfg.VIDEOS and cfg.VIDEOS[0].startswith("http") else None
print(ficha if ficha else "No hay enlace web configurado.")


In [ ]:
if ficha is not None:
    ruta_video = cfg.DIR_VIDEO / ficha["archivo"]
    print(f"Archivo listo: {ruta_video.name} ? {ruta_video.stat().st_size / 1e6:.1f} MB")
else:
    ruta_video = None
    print("No se descarg? ning?n v?deo en esta prueba.")


Si una fuente no est? disponible, no intentes eludir controles de acceso. Usa un archivo local cuyo uso est? permitido o revisa las condiciones del proveedor.


In [ ]:
def extraer_audio(ruta_video, ruta_audio, frecuencia=16_000):
    """Extrae una pista mono WAV a la frecuencia indicada."""
    orden = ["ffmpeg", "-y", "-loglevel", "error", "-i", str(ruta_video),
             "-vn", "-ac", "1", "-ar", str(frecuencia), "-c:a", "pcm_s16le", str(ruta_audio)]
    subprocess.run(orden, check=True)
    return ruta_audio

if ruta_video is not None:
    ruta_audio = extraer_audio(ruta_video, cfg.DIR_AUDIO / f"{ficha['id']}.wav")
    print("Creado:", ruta_audio.relative_to(cfg.RAIZ))
else:
    ruta_audio = None


## Extraer y revisar el audio

La conversi?n guarda una pista WAV mono a 16 kHz, adecuada para an?lisis de voz. La celda siguiente informa duraci?n y muestra una se?al b?sica; escucha el resultado antes de continuar.


In [ ]:
if ruta_audio is not None:
        import numpy as np
        import soundfile as sf
        import matplotlib.pyplot as plt

        audio, fs = sf.read(ruta_audio, dtype="float32")
        print(f"Muestras: {len(audio):,} · frecuencia: {fs} Hz · canales: {audio.ndim} · duración: {len(audio) / fs / 60:.1f} min")
        print(f"Vídeo: {ruta_video.stat().st_size / 1e6:.1f} MB   ·   Audio WAV: {ruta_audio.stat().st_size / 1e6:.1f} MB")

        # Volumen máximo de cada segundo: un dibujo rápido de dónde hay sonido y dónde silencio
        segundos = len(audio) // fs
        volumen = np.abs(audio[: segundos * fs]).reshape(segundos, fs).max(axis=1)
        plt.figure(figsize=(11, 2.2))
        plt.fill_between(np.arange(segundos) / 60, volumen, color="#4C72B0")
        plt.xlabel("minuto"); plt.ylabel("volumen"); plt.title(ficha["titulo"]); plt.margins(x=0)
        plt.show()
else:
    print("No hay audio que revisar en esta prueba.")


In [ ]:
if ruta_audio is not None:
    from IPython.display import Audio, display
    display(Audio(audio[: 20 * fs], rate=fs))
else:
    print("A?ade una fuente permitida para escuchar el audio.")


## Procesar la lista configurada

La celda genera un WAV por cada fuente y un ?ndice `datos/videos.json`. Si la lista est? vac?a, el cuaderno termina sin descargar nada. Los archivos de entrada y salida est?n ignorados por Git.


In [ ]:
def duracion(ruta):
    """Duración en segundos de cualquier archivo de audio o vídeo (se la preguntamos a ffprobe)."""
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(ruta)],
                       capture_output=True, text=True, check=True)
    return float(r.stdout)

def usar_archivo_local(ruta):
    """Incorpora al proyecto un archivo que ya tienes (vídeo o audio) como si viniera de YouTube."""
    ruta = Path(ruta).expanduser()
    destino = cfg.DIR_VIDEO / ruta.name
    if not destino.exists():
        shutil.copy(ruta, destino)
    return {"id": ruta.stem, "titulo": ruta.stem, "canal": None, "duracion_s": round(duracion(destino)),
            "licencia": "archivo propio", "origen": str(ruta), "archivo": destino.name}

fichas = []
for origen in cfg.VIDEOS:
    es_enlace = origen.startswith("http")
    ficha = descargar_video(origen) if es_enlace else usar_archivo_local(origen)
    wav = cfg.DIR_AUDIO / f"{ficha['id']}.wav"
    if not wav.exists():
        extraer_audio(cfg.DIR_VIDEO / ficha["archivo"], wav)
    fichas.append(ficha)
    print(f"✅ {ficha['id']:14} {ficha['duracion_s'] / 60:5.1f} min · {ficha['titulo']}")

cfg.FICHA_VIDEOS.parent.mkdir(parents=True, exist_ok=True)
cfg.FICHA_VIDEOS.write_text(json.dumps(fichas, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nÍndice guardado en {cfg.FICHA_VIDEOS.relative_to(cfg.RAIZ)} · total: {len(fichas)} archivos")

In [ ]:
pd.DataFrame(fichas, columns=["id", "titulo", "canal", "duracion_s", "licencia"])


## Prueba local

A?ade una URL permitida o la ruta a un archivo local en tu `config.py`, vuelve al inicio y ejecuta las celdas en orden. Confirma que el audio se entiende y que el archivo corresponde a la fuente elegida.


In [ ]:
if cfg.VIDEOS:
    print(f"Proceso completado: {len(fichas)} archivo(s) listos en {cfg.DIR_AUDIO}")
else:
    print("Configura una fuente autorizada en config.py y ejecuta de nuevo las celdas de descarga.")


El resultado queda en `datos/audio/`. Este repositorio no incluye reconocimiento de voz; puedes usar el WAV en otro flujo de trabajo compatible, respetando las licencias y el consentimiento de las personas grabadas.
